# Podcast Processor — Walkthrough

This notebook walks through the **podcast processing pipeline**: from an RSS feed to a structured markdown file with transcript and summary (Obsidian-ready).

**Pipeline steps:**
1. **RSS parsing** — Get latest episode metadata (title, description, audio URL)
2. **Audio download** — Download the MP3 to a temp directory
3. **Transcription** — AssemblyAI (with diarisation) or OpenAI Whisper (fallback)
4. **Speaker attribution** — Claude maps "Speaker A/B" to real names using episode description
5. **Summary** — Claude produces: narrative summary, themes, trade recommendations, notable quotes
6. **Output** — Save markdown with YAML frontmatter and full transcript

**Requirements:** Set `ANTHROPIC_API_KEY` in `.env` for speaker attribution and summary. For transcription use either `ASSEMBLYAI_API_KEY` (preferred) or `OPENAI_API_KEY`.

## Setup

1. **Select the project kernel** so `dotenv` and dependencies are available: click the kernel name (top right) → **Select Another Kernel** → choose **"Python (Experimental-Sandbox .venv)"** (or pick the interpreter at `Experimental-Sandbox/.venv/bin/python`).
2. Run the cell below. It finds the project root, loads `.env`, and imports the pipeline modules.

In [2]:
import os
import sys
from pathlib import Path

# Find project root (directory containing podcast_processor/ and notebooks/)
# Use the kernel that has project deps: select ".venv (Experimental-Sandbox)" in the notebook kernel picker.
ROOT = Path.cwd()
if (ROOT / "Experimental-Sandbox" / "podcast_processor").is_dir():
    ROOT = ROOT / "Experimental-Sandbox"
elif (ROOT / "podcast_processor").is_dir():
    pass
elif ROOT.name == "notebooks" and (ROOT.parent / "podcast_processor").is_dir():
    ROOT = ROOT.parent
else:
    for parent in ROOT.parents:
        if (parent / "podcast_processor").is_dir():
            ROOT = parent
            break
        if (parent / "Experimental-Sandbox" / "podcast_processor").is_dir():
            ROOT = parent / "Experimental-Sandbox"
            break
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

from dotenv import load_dotenv
# Load .env from project root; also try parent workspace .env if you keep keys there
load_dotenv(ROOT / ".env")
if not os.environ.get("ANTHROPIC_API_KEY") and (ROOT.parent / ".env").exists():
    load_dotenv(ROOT.parent / ".env")

from podcast_processor.rss_parser import EpisodeMetadata, get_latest_episode, parse_feed
from podcast_processor.audio_download import download_audio
from podcast_processor.transcription import transcribe, TranscriptResult, Utterance
from podcast_processor.speaker_attribution import attribute_speakers
from podcast_processor.summary import generate_summary, EpisodeSummary
from podcast_processor.output import save_markdown

print("Project root:", ROOT)
print("ANTHROPIC_API_KEY set:", bool(os.environ.get("ANTHROPIC_API_KEY")))
print("ASSEMBLYAI_API_KEY set:", bool(os.environ.get("ASSEMBLYAI_API_KEY")))
print("OPENAI_API_KEY set:", bool(os.environ.get("OPENAI_API_KEY")))

ModuleNotFoundError: No module named 'dotenv'

---
## Step 1: Parse RSS feed

The pipeline uses **feedparser** to read the podcast RSS. Default feed: *At Any Rate* (Podbean). We get the **latest episode** and its metadata: title, published date, description, and **audio URL** (enclosure).

In [ ]:
DEFAULT_FEED_URL = "https://feed.podbean.com/atanyrate/feed.xml"

# Get latest episode only
episode = get_latest_episode(DEFAULT_FEED_URL)

print("Title:", episode.title)
print("Published:", episode.published)
print("Audio URL:", episode.audio_url[:80] + "..." if len(episode.audio_url) > 80 else episode.audio_url)
print("Description (first 200 chars):", (episode.description or "")[:200], "...")

In [ ]:
# Optional: list last N episodes (parse_feed returns newest first)
all_episodes = parse_feed(DEFAULT_FEED_URL)
for i, ep in enumerate(all_episodes[:5], 1):
    print(f"{i}. {ep.title[:60]}..." if len(ep.title) > 60 else f"{i}. {ep.title}")

---
## Step 2: Download audio

**audio_download** uses `requests` to stream the MP3 to a temp directory. Returns the local `Path`. Large files may take a minute.

In [ ]:
import tempfile

tmp_dir = tempfile.mkdtemp(prefix="podcast_notebook_")
audio_path = download_audio(episode.audio_url, dest_dir=tmp_dir)
print("Downloaded to:", audio_path)
print("Size (MB):", round(audio_path.stat().st_size / 1e6, 2))

---
## Step 3: Transcription

- **AssemblyAI** (if `ASSEMBLYAI_API_KEY` is set): transcription + **speaker diarisation** → each segment has a speaker label (Speaker A, B, …) and timestamps.
- **Whisper** (if only `OPENAI_API_KEY`): transcript text only, no speaker labels.

Result is a **TranscriptResult**: `utterances` (list of `Utterance`: speaker, text, start_ms, end_ms), `raw_text`, `has_diarisation`, `source`.

In [ ]:
transcript = transcribe(audio_path)

print("Source:", transcript.source)
print("Has diarisation:", transcript.has_diarisation)
print("Number of utterances:", len(transcript.utterances))
print("Raw text length (chars):", len(transcript.raw_text))
if transcript.utterances:
    print("\nFirst 5 utterances:")
    for u in transcript.utterances[:5]:
        print(f"  {u.speaker}: {u.text[:80]}..." if len(u.text) > 80 else f"  {u.speaker}: {u.text}")

---
## Step 4: Speaker attribution

**Claude** uses the episode description (and a sample of the transcript) to map generic labels like "Speaker A" / "Speaker B" to real names (e.g. host and guest). If there was no diarisation (Whisper path), Claude attributes the full transcript to speakers in one go.

Requires **ANTHROPIC_API_KEY**.

In [ ]:
anthropic_key = os.environ.get("ANTHROPIC_API_KEY")
if not anthropic_key:
    print("ANTHROPIC_API_KEY not set — skipping speaker attribution. Use a mock transcript for demo.")
    # Build a minimal mock so we can still show the rest of the pipeline
    transcript.utterances = [
        Utterance(speaker="Host", text="Today we're discussing the Fed and rate cuts."),
        Utterance(speaker="Guest", text="I think we'll see two cuts by year end."),
    ]
    transcript.has_diarisation = True
else:
    transcript = attribute_speakers(
        transcript,
        episode_description=episode.description or "",
        anthropic_api_key=anthropic_key,
    )

speakers = sorted({u.speaker for u in transcript.utterances})
print("Identified speakers:", speakers)

---
## Step 5: Summary generation

**Claude** produces a structured **EpisodeSummary**: narrative summary, key themes, trade recommendations / market views, and notable quotes. The prompt is tuned for a markets/finance podcast.

In [ ]:
if anthropic_key:
    summary = generate_summary(
        transcript,
        episode_title=episode.title,
        anthropic_api_key=anthropic_key,
    )
else:
    summary = EpisodeSummary(
        summary="Demo summary: Fed and rate cuts discussed.",
        themes=["Monetary policy", "Rate cuts"],
        trade_recommendations=["Two cuts by year end"],
        notable_quotes=["Guest: I think we'll see two cuts by year end."],
    )

print("Summary (first 300 chars):", summary.summary[:300], "...")
print("\nThemes:", summary.themes)
print("Trade recommendations:", summary.trade_recommendations)
print("Notable quotes:", summary.notable_quotes[:2])

---
## Step 6: Save markdown output

**output.save_markdown** writes a single markdown file with:
- YAML frontmatter (title, date, author, audio_url, transcription_source, themes, tags)
- Episode header and description
- Summary, Key Themes, Trade Recommendations, Notable Quotes
- Full transcript with speaker labels

Designed for **Obsidian** (tags and frontmatter). Output directory defaults to `./output` or `OUTPUT_DIR`.

In [ ]:
out_dir = os.environ.get("OUTPUT_DIR", "./output")
filepath = save_markdown(episode, transcript, summary, output_dir=out_dir)
print("Saved to:", filepath)
print("\nFirst 50 lines of generated file:")
print("-" * 40)
with open(filepath, encoding="utf-8") as f:
    for i, line in enumerate(f):
        if i >= 50:
            print("...")
            break
        print(line, end="")

---
## Running the full pipeline from the CLI

From the project root:

```bash
python -m podcast_processor.pipeline --feed-url "https://feed.podbean.com/atanyrate/feed.xml" --output-dir ./output -v
```

Or install the project and run the entry point:

```bash
pip install -e .
podcast-processor --feed-url "..." --output-dir ./output -v
```

Ensure `.env` contains at least `ANTHROPIC_API_KEY` and one of `ASSEMBLYAI_API_KEY` or `OPENAI_API_KEY`.